# Convex Here, Not Convex There

**Topic:** Convexity &nbsp;·&nbsp; **Difficulty:** Medium

"Is this function convex?" is not a yes/no question until you say *where*. A
function can satisfy the definition on one region and violate it a short distance
away, and gradient descent will behave completely differently in the two places.

Test three candidate functions numerically (the chord test on random pairs and
the Hessian eigenvalue test on random points) and find which two are convex
everywhere and which one is convex only outside a strip through the origin.

## The data (created in the next cell)

- Three scalar functions of a 2-vector: `f1` (a quadratic), `f2` (log-sum-exp),
  `f3` (a quartic with a saddle).
- `pairs`: 400 random pairs of points for the chord test.
- `points`: 400 random points spread widely, plus `points_near`, 400 more from
  the tight Gaussian `N(0, 0.15)` around the origin.

## Your task

With **NumPy only**:

1. Chord test: `f(t*x + (1-t)*y) <= t*f(x) + (1-t)*f(y)` for `t = 0.5`. Count
   violations (tolerance `1e-9`) over `pairs` for each function.
2. Hessian test: build the 2x2 Hessian by central differences (`h = 1e-4`) and
   record its smallest eigenvalue. Count points where it is below `-1e-6`.
3. Do the Hessian test on both `points` (wide) and `points_near` (the
   tight Gaussian around the origin).
4. Set `answers` to a dict with `"chord_viol"` (length-3 ints),
   `"hess_viol_wide"` (length-3), `"hess_viol_near"` (length-3), and
   `"min_eig_f3_wide"`: the smallest eigenvalue of `f3`'s Hessian over the wide
   points.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks `f1` and `f2` pass at every sampled point, while `f3` fails at every point near the
origin and at only a minority of the widely-spread points.

In [ ]:
import numpy as np

rng = np.random.default_rng(3005)

def f1(z):
    return 2.0 * z[0] ** 2 + 3.0 * z[1] ** 2 + z[0] * z[1]

def f2(z):
    a = np.array([z[0], z[1], 0.0])
    return float(np.log(np.sum(np.exp(a - a.max()))) + a.max())

def f3(z):
    return z[0] ** 4 - 3.0 * z[0] ** 2 + z[1] ** 2

funcs = [f1, f2, f3]
pairs = rng.normal(0, 2.0, (400, 2, 2))
points = rng.normal(0, 2.0, (400, 2))
points_near = rng.normal(0, 0.15, (400, 2))
print("three candidates: a quadratic, log-sum-exp, and a quartic with a saddle")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("chord_viol", [3], 40.0, -60.65323721926545, 1e-06),
    ("hess_viol_wide", [3], 108.0, -163.7637404920167, 1e-06),
    ("hess_viol_near", [3], 400.0, -606.5323721926544, 1e-06),
    ("min_eig_f3_wide", [], -5.999984870053154, 1.004625555558731, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "hess_viol_wide" in answers:
    _w = np.asarray(answers["hess_viol_wide"], dtype=float)
    _n = np.asarray(answers["hess_viol_near"], dtype=float)
    _good = (_w[0] == 0 and _w[1] == 0 and _n[0] == 0 and _n[1] == 0
             and _n[2] == 400 and 0 < _w[2] < 400)
    print(f"  {'f3 non-convex at origin':<24} {'ok' if _good else 'wrong'}  "
          f"(wide violations {_w.astype(int).tolist()}, near {_n.astype(int).tolist()})")
    _ok = _ok and _good

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

`f3 = x^4 - 3x^2 + y^2` has second derivative `12x^2 - 6` in `x`, which is
negative whenever `|x| < 0.707`, so it is concave-in-`x` inside a strip through
the origin and convex outside it. Every point in the tight Gaussian
cluster violates, but only
the ~27% of the wide points that land inside the strip do. The chord test with
`t = 0.5` catches a pair only when the midpoint sits in the bad region, so it
sees fewer violations than the Hessian test. The
chord test is a *necessary* condition sampled at finitely many pairs, so passing
it is evidence rather than proof, and the Hessian test is the one with teeth.

```python
import numpy as np

def chord_violations(f):
    c = 0
    for x, y in pairs:
        mid = 0.5 * x + 0.5 * y
        if f(mid) > 0.5 * f(x) + 0.5 * f(y) + 1e-9:
            c += 1
    return int(c)

def hessian(f, z, h=1e-4):
    H = np.zeros((2, 2))
    for i in range(2):
        for j in range(2):
            ei = np.zeros(2); ei[i] = h
            ej = np.zeros(2); ej[j] = h
            H[i, j] = (f(z + ei + ej) - f(z + ei - ej)
                       - f(z - ei + ej) + f(z - ei - ej)) / (4 * h * h)
    return (H + H.T) / 2

def hess_violations(f, pts):
    c = 0
    worst = np.inf
    for z in pts:
        w = np.linalg.eigvalsh(hessian(f, z))
        worst = min(worst, float(w.min()))
        if w.min() < -1e-6:
            c += 1
    return int(c), worst

chord_viol = np.array([chord_violations(f) for f in funcs], dtype=float)

wide, near, worst_f3 = [], [], None
for idx, f in enumerate(funcs):
    cw, w = hess_violations(f, points)
    cn, _ = hess_violations(f, points_near)
    wide.append(cw)
    near.append(cn)
    if idx == 2:
        worst_f3 = w

answers = {
    "chord_viol": chord_viol,
    "hess_viol_wide": np.array(wide, dtype=float),
    "hess_viol_near": np.array(near, dtype=float),
    "min_eig_f3_wide": float(worst_f3),
}
```